# Asyncio & Concurrency — Fast I/O

In [ ]:
# Day 37 — Asyncio & Concurrency
#
# Imagine a chef cooking a meal.
# SYNCHRONOUS: The chef puts water on the stove and STARES AT IT until it boils.
#              Only then do they start chopping the vegetables.
# ASYNCHRONOUS: The chef puts water on the stove, and WHILE waiting, chops the veggies.
#
# In programming, waiting for water to boil = waiting for a network request, a database
# query, or reading a file (I/O Bound tasks).
# 
# If your Python script needs to query the status of 100 Azure servers, a synchronous
# script will wait for server 1, then server 2... taking forever.
# Async Python can query all 100 at the same time and process whichever returns first.
#
# Today we'll cover:
#   1. `async def` and `await`
#   2. Event Loops (`asyncio.run`)
#   3. Running tasks concurrently (`asyncio.gather`)
#   4. Async HTTP requests with `httpx` (or `aiohttp`)
#
# Install: pip install httpx

### 1. The Basics of Async and Await

In [ ]:
import asyncio
import time

# This is a "coroutine" - a function that can pause and yield control
async def boil_water():
    print("Starting to boil water...")
    # await pauses this function and lets the event loop do other things
    # asyncio.sleep simulates a non-blocking wait (like a network call)
    await asyncio.sleep(2) 
    print("Water is boiling!")
    return "Boiled Water"

async def chop_veggies():
    print("Starting to chop veggies...")
    await asyncio.sleep(1)
    print("Veggies are chopped!")
    return "Chopped Veggies"

# We need a main coroutine to orchestrate
async def main():
    start_time = time.time()
    
    # If we just 'await' them in order, it's still sequential!
    # print("Sequential:")
    # await boil_water() # Takes 2 secs
    # await chop_veggies() # Takes 1 sec
    # Total: 3 seconds

    # To run concurrently, we use asyncio.gather
    print("Concurrent:")
    results = await asyncio.gather(
        boil_water(),
        chop_veggies()
    )
    
    print(f"Finished in {time.time() - start_time:.2f} seconds")
    print(f"Results: {results}")

# How to actually run async code:
# In standard Python scripts you use: asyncio.run(main())
# But Jupyter Notebooks ALREADY run inside an async event loop!
# So in Jupyter, we can just await the main function directly:
await main()

### 2. Real World: Async HTTP Requests

In [ ]:
# The standard `requests` library is SYNCHRONOUS.
# If you use `requests.get()` inside an async function, it blocks the whole loop!
# For async, we use `httpx`.

import httpx

async def fetch_pokemon(client, id):
    print(f"Fetching Pokemon {id}...")
    response = await client.get(f"https://pokeapi.co/api/v2/pokemon/{id}")
    data = response.json()
    print(f"Got Pokemon {id}: {data['name']}")
    return data['name']

async def fetch_all():
    start_time = time.time()
    
    # We use an AsyncClient context manager to manage connections
    async with httpx.AsyncClient() as client:
        # Create a list of tasks (we don't await them yet!)
        tasks = [fetch_pokemon(client, i) for i in range(1, 11)]
        
        # Run all 10 requests at exactly the same time
        results = await asyncio.gather(*tasks)
        
    print(f"\nFetched {len(results)} Pokemon concurrently in {time.time() - start_time:.2f} seconds")
    print(results)

await fetch_all()

### 3. CPU-Bound vs I/O-Bound

In [ ]:
# CRITICAL ML CONCEPT:
# Asyncio is amazing for I/O Bound tasks (Network, Database, File Reads).
# It is TERRIBLE for CPU Bound tasks (Math, ML Training, Image Processing).
# 
# Why? Because Python has the GIL (Global Interpreter Lock).
# If you run a heavy ML model prediction inside an async function without yielding,
# no other async task can run. It blocks the event loop.
#
# Rule of Thumb:
# - Call an external API -> use `asyncio`
# - Train a Random Forest -> use `multiprocessing`

print("Remember: Don't put heavy ML processing directly in an async endpoint.")
print("In FastAPI, if you have a heavy ML function, define it as 'def' instead of 'async def',")
print("and FastAPI will automatically run it in a separate thread pool!")

In [ ]:
# Practice Questions:
#
# 1. Write an async function that simulates "downloading" 3 files of different sizes.
#    Use asyncio.sleep(size) to simulate the download time. Run them concurrently.
#
# 2. Why should you NOT use the standard `requests` library in FastAPI async endpoints?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: Async Health Checker
# You have a list of 5 server URLs (you can mock them). Write an async function that uses
# httpx to hit their /healthz endpoints concurrently. If one takes more than 2 seconds,
# it should timeout (look up `asyncio.wait_for`).

# Why this matters for me as a DevOps/MLOps engineer:
# 1. Scaling: Async allows a single Python process to handle thousands of connections.
# 2. Automation: When triggering builds or deployments across 100s of clusters, you must do it asynchronously.
# 3. FastAPI Core: Understanding async is required to write performant FastAPI model servers.